In [1]:
!nvidia-smi

Sat May  2 11:17:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!pip install --upgrade pip
!pip install "vllm>=0.19.1"
!pip install "lm_eval[vllm,wandb,multilingual,math,ifeval]"
!pip install wandb huggingface_hub
 
# Verify GPU + CUDA + disk
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')"
!df -h /content   # confirm >70GB free for the model download

name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
PyTorch 2.11.0+cu130, CUDA 13.0
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   54G  183G  23% /


In [5]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
RESULTS_DIR = '/content/drive/MyDrive/resilient_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
login(hf_token)  # paste your HF token

wandb_api = os.environ.get("WANDB_API_KEY") 
import wandb
wandb.login(key=wandb_api)  # paste your W&B token

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saesha-parekh (saesha-parekhcivicdatalab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [17]:
import subprocess, os, shutil

# Clear stale compiled cache that embeds the broken FlashInfer call
shutil.rmtree("/root/.cache/vllm/torch_compile_cache", ignore_errors=True)

os.environ["VLLM_USE_FLASHINFER_MOE_FP16"] = "0"
os.environ["VLLM_USE_FLASHINFER_MOE_FP8"] = "0"
os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"
os.environ["FLASHINFER_CUDA_ARCH_LIST"] = "12.0f"  # Blackwell "final" arch suffix

vllm_proc = subprocess.Popen([
    "vllm", "serve", "sarvamai/sarvam-30b",
    "--dtype", "bfloat16",
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "16384",
    "--enable-chunked-prefill",
    "--enforce-eager",   # disables torch.compile + CUDA graphs → uses Triton MoE
    "--trust-remote-code",
    "--port", "8000",
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ})

for line in vllm_proc.stdout:
    print(line, end='')
    if "Application startup complete" in line or "Uvicorn running" in line:
        print("\n✅ vLLM server ready")
        break

(APIServer pid=16132) INFO 05-02 12:04:09 [utils.py:299] 
(APIServer pid=16132) INFO 05-02 12:04:09 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=16132) INFO 05-02 12:04:09 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.20.0
(APIServer pid=16132) INFO 05-02 12:04:09 [utils.py:299]   █▄█▀ █     █     █     █  model   sarvamai/sarvam-30b
(APIServer pid=16132) INFO 05-02 12:04:09 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=16132) INFO 05-02 12:04:09 [utils.py:299] 
(APIServer pid=16132) INFO 05-02 12:04:09 [utils.py:233] non-default args: {'model_tag': 'sarvamai/sarvam-30b', 'model': 'sarvamai/sarvam-30b', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 16384, 'enforce_eager': True, 'gpu_memory_utilization': 0.9, 'enable_chunked_prefill': True}
(APIServer pid=16132) INFO 05-02 12:04:10 [model.py:555] Resolved architecture: SarvamMoEForCausalLM
(APIServer pid=16132) INFO 05-02 12:04:10 [model.py:2015] Downcasting torch.float32 to torch.bfloat1

In [18]:
import requests
r = requests.post(
    "http://localhost:8000/v1/completions",
    json={"model": "sarvamai/sarvam-30b",
          "prompt": "The capital of India is",
          "max_tokens": 10}
)
print(r.json())

{'id': 'cmpl-95556ed85503deb6', 'object': 'text_completion', 'created': 1777723998, 'model': 'sarvamai/sarvam-30b', 'choices': [{'index': 0, 'text': ' is is is is is is is is is is', 'logprobs': None, 'finish_reason': 'length', 'stop_reason': None, 'token_ids': None, 'prompt_logprobs': None, 'prompt_token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 6, 'total_tokens': 16, 'completion_tokens': 10, 'prompt_tokens_details': None}, 'kv_transfer_params': None}
